In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from src import synthetic, features as F

In [2]:
FORECAST_HORIZON = 10
NUM_STORES = 4
NUM_DAYS = 365*4

# Generate sales features

In [3]:

stores = synthetic.store_data(n_stores=NUM_STORES)
sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)
sales_stores = sales.merge(stores, on='Store', how='left').set_index(['Store', 'Date'])
sales_stores = sales_stores.sample(frac=1, random_state=42)  # Shuffle the data

no_state_holiday = sales_stores['StateHoliday'].isin(['0', 0]) | sales_stores['StateHoliday'].isna()
sales_stores.loc[no_state_holiday, 'StateHoliday'] = 'NoHoliday'
sales_stores['isStateHoliday'] = sales_stores['StateHoliday'] != 'NoHoliday'
sales_stores['SchoolHoliday'] = sales_stores['SchoolHoliday'].astype(bool)

sales_stores.loc[sales_stores['Sales'] == 0, 'Sales'] = np.nan
sales_stores.ffill(inplace=True)  # Fill NaN values with the last valid observation



In [4]:
sales_stores['isStateHoliday'].any()

np.True_

In [5]:
""" Past features """

lags = [pd.DateOffset(days=i) for i in range(1, 3)]
diffs = [pd.DateOffset(days=i) for i in range(1, 3)]
windows = ['7D', '14D', '30D']


one_day_offset = pd.DateOffset(days=1)
forecast_offset = pd.DateOffset(days=FORECAST_HORIZON)


grouped = sales_stores.reset_index('Store').groupby('Store')

x_lag = grouped['Sales'].apply(lambda x: F.lags(x, lags))
x_dif = grouped['Sales'].apply(lambda x: F.diffs(x, diffs, one_day_offset))
x_window = grouped['Sales'].apply(lambda x: F.rolling(x, windows, 'mean', one_day_offset))
x_calendar = grouped['Sales'].apply(lambda x: F.calendar(x.index.to_series()+forecast_offset))

x_competition = grouped['CompetitionSinceDate'].apply(lambda x: F.competition_since_days(x, forecast_offset))
x_state_hol = grouped['isStateHoliday'].apply(lambda x: F.holiday_counters(x, forecast_offset))
x_school_hol = grouped['SchoolHoliday'].apply(lambda x: F.holiday_counters(x, forecast_offset))

x_synth = pd.concat([x_lag, x_dif, x_window, x_calendar, x_competition, x_state_hol, x_school_hol], axis=1)

x_synth.sort_index().head(10)

lag_days_1   lag_days_2  diff_days_1  diff_days_2  \
Store Date                                                             
1     2013-01-01          NaN          NaN          NaN          NaN   
      2013-01-02   467.320508          NaN          NaN          NaN   
      2013-01-03   467.820508   467.320508     0.500000          NaN   
      2013-01-04   451.000000   467.820508   -16.820508   -16.320508   
      2013-01-05   434.179492   451.000000   -16.820508   -33.641016   
      2013-01-06   434.679492   434.179492     0.500000   -16.320508   
      2013-01-07  2707.679492   434.679492  2273.000000  2273.500000   
      2013-01-08   453.000000  2707.679492 -2254.679492    18.320508   
      2013-01-09   470.820508   453.000000    17.820508 -2236.858984   
      2013-01-10   471.320508   470.820508     0.500000    18.320508   

                  rolling_mean_7D  rolling_mean_14D  rolling_mean_30D  year  \
Store Date                                                                    
1     2013-01-01              NaN               NaN               NaN  2013   
      2013-01-02       467.320508        467.320508        467.320508  2013   
      2013-01-03       467.570508        467.570508        467.570508  2013   
      2013-01-04       462.047005        462.047005        462.047005  2013   
      2013-01-05       455.080127        455.080127        455.080127  2013   
      2013-01-06       451.000000        451.000000        451.000000  2013   
      2013-01-07       827.113249        827.113249        827.113249  2013   
      2013-01-08       773.668499        773.668499        773.668499  2013   
      2013-01-09       774.168499        735.812500        735.812500  2013   
      2013-01-10       774.668499        706.424501        706.424501  2013   

                  quarter  month  ...  day_of_week  is_month_start  \
Store Date                        ...                                
1     2013-01-01        1      1  ...            4           False   
      2013-01-02        1      1  ...            5           False   
      2013-01-03        1      1  ...            6           False   
      2013-01-04        1      1  ...            0           False   
      2013-01-05        1      1  ...            1           False   
      2013-01-06        1      1  ...            2           False   
      2013-01-07        1      1  ...            3           False   
      2013-01-08        1      1  ...            4           False   
      2013-01-09        1      1  ...            5           False   
      2013-01-10        1      1  ...            6           False   

                  is_weekend  is_weekday  is_month_end  \
Store Date                                               
1     2013-01-01       False        True         False   
      2013-01-02        True       False         False   
      2013-01-03        True       False         False   
      2013-01-04       False        True         False   
      2013-01-05       False        True         False   
      2013-01-06       False        True         False   
      2013-01-07       False        True         False   
      2013-01-08       False        True         False   
      2013-01-09        True       False         False   
      2013-01-10        True       False         False   

                  CompetitionDaysSinceStart  DaysToNextHoliday  \
Store Date                                                       
1     2013-01-01                        285               18.0   
      2013-01-02                        286               17.0   
      2013-01-03                        287               16.0   
      2013-01-04                        288               15.0   
      2013-01-05                        289               14.0   
      2013-01-06                        290               13.0   
      2013-01-07                        291               12.0   
      2013-01-08                        292               11.0   
      2

In [6]:
from src import constants as C


sales = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
stores = pd.read_csv(C.STORE_FILE)
sales_stores = sales.merge(stores, on='Store', how='left').set_index(['Store', 'Date'])

no_state_holiday = sales_stores['StateHoliday'].isin(['0', 0]) | sales_stores['StateHoliday'].isna()
sales_stores.loc[no_state_holiday, 'StateHoliday'] = 'NoHoliday'
sales_stores['isStateHoliday'] = sales_stores['StateHoliday'] != 'NoHoliday'
sales_stores['isSchoolHoliday'] = sales_stores['SchoolHoliday'].astype(bool)

sales_stores['CompetitionStartDate'] = pd.to_datetime(
    sales_stores[['CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth']]
    .rename(columns={'CompetitionOpenSinceYear': 'year', 'CompetitionOpenSinceMonth': 'month'})
    .assign(day=1)
)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_19636\2969052399.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [8]:
grouped = sales_stores.reset_index('Store').groupby('Store')

x_lag = grouped['Sales'].apply(lambda x: F.lags(x, lags))
x_dif = grouped['Sales'].apply(lambda x: F.diffs(x, diffs, one_day_offset))
x_window = grouped['Sales'].apply(lambda x: F.rolling(x, windows, 'mean', one_day_offset))
x_calendar = grouped['Sales'].apply(lambda x: F.calendar(x.index.to_series()+forecast_offset))

x_competition = grouped['CompetitionStartDate'].apply(lambda x: F.competition_since_days(x, forecast_offset))
x_state_hol = grouped['isStateHoliday'].apply(lambda x: F.holiday_counters(x, forecast_offset))
x_school_hol = grouped['SchoolHoliday'].apply(lambda x: F.holiday_counters(x, forecast_offset))

x_all = pd.concat([x_lag, x_dif, x_window, x_calendar, x_competition, x_state_hol, x_school_hol], axis=1)

x_all.sort_index().head(10)

lag_days_1  lag_days_2  diff_days_1  diff_days_2  \
Store Date                                                           
1     2013-01-01         NaN         NaN          NaN          NaN   
      2013-01-02         0.0         NaN          NaN          NaN   
      2013-01-03      5530.0         0.0       5530.0          NaN   
      2013-01-04      4327.0      5530.0      -1203.0       4327.0   
      2013-01-05      4486.0      4327.0        159.0      -1044.0   
      2013-01-06      4997.0      4486.0        511.0        670.0   
      2013-01-07         0.0      4997.0      -4997.0      -4486.0   
      2013-01-08      7176.0         0.0       7176.0       2179.0   
      2013-01-09      5580.0      7176.0      -1596.0       5580.0   
      2013-01-10      5471.0      5580.0       -109.0      -1705.0   

                  rolling_mean_7D  rolling_mean_14D  rolling_mean_30D  year  \
Store Date                                                                    
1     2013-01-01              NaN               NaN               NaN  2013   
      2013-01-02         0.000000          0.000000          0.000000  2013   
      2013-01-03      2765.000000       2765.000000       2765.000000  2013   
      2013-01-04      3285.666667       3285.666667       3285.666667  2013   
      2013-01-05      3585.750000       3585.750000       3585.750000  2013   
      2013-01-06      3868.000000       3868.000000       3868.000000  2013   
      2013-01-07      3223.333333       3223.333333       3223.333333  2013   
      2013-01-08      3788.000000       3788.000000       3788.000000  2013   
      2013-01-09      4585.142857       4012.000000       4012.000000  2013   
      2013-01-10      4576.714286       4174.111111       4174.111111  2013   

                  quarter  month  ...  day_of_week  is_month_start  \
Store Date                        ...                                
1     2013-01-01        1      1  ...            4           False   
      2013-01-02        1      1  ...            5           False   
      2013-01-03        1      1  ...            6           False   
      2013-01-04        1      1  ...            0           False   
      2013-01-05        1      1  ...            1           False   
      2013-01-06        1      1  ...            2           False   
      2013-01-07        1      1  ...            3           False   
      2013-01-08        1      1  ...            4           False   
      2013-01-09        1      1  ...            5           False   
      2013-01-10        1      1  ...            6           False   

                  is_weekend  is_weekday  is_month_end  \
Store Date                                               
1     2013-01-01       False        True         False   
      2013-01-02        True       False         False   
      2013-01-03        True       False         False   
      2013-01-04       False        True         False   
      2013-01-05       False        True         False   
      2013-01-06       False        True         False   
      2013-01-07       False        True         False   
      2013-01-08       False        True         False   
      2013-01-09        True       False         False   
      2013-01-10        True       False         False   

                  CompetitionDaysSinceStart  DaysToNextHoliday  \
Store Date                                                       
1     2013-01-01                     1593.0               77.0   
      2013-01-02                     1594.0               76.0   
      2013-01-03                     1595.0               75.0   
      2013-01-04                     1596.0               74.0   
      2013-01-05                     1597.0               73.0   
      2013-01-06                     1598.0               72.0   
      2013-01-07                     1599.0               71.0   
      2013-01-08                     1600.0               70.0   
      2013-01-09              